# 🏨 Hotel do Mosquito — Testes de Banco de Dados

**Laboratório de Banco de Dados**  
Camili de Moura Marangoni · Lucas Sobrinho Santos · Maria Eduarda Patu Ângelo da Silva  
Matheus Pinheiro de Camargo Silva · Willian Alexandre Schwingel Ferreira

---

> **ℹ️ Como usar este notebook:**  
> Execute as células **em ordem**, uma por vez.  
> Clique no botão ▶ à esquerda de cada célula e aguarde o ✅ aparecer antes de passar para a próxima.
>
> A **Célula 1** demora ~1 minuto (instala MySQL e clona o repositório).  
> As demais são rápidas.

---

### O que este notebook testa

| # | Teste | Elemento do BD |
|---|---|---|
| 1 | Autenticação com credenciais corretas e erradas | `sp_Login` |
| 2 | Fluxo completo: reserva → check-in → consumo → check-out | 4 SPs transacionais |
| 3 | Sobreposição de reservas no mesmo quarto | `sp_RegistrarReserva` |
| 4 | Histórico de preço gerado automaticamente | Triggers |
| 5 | ROLLBACK desfaz trigger | Transactions |
| 6 | SAVEPOINT com rollback parcial | Transactions |
| 7 | Bloqueio de Recepcionista no banco | `sp_AssertGerente` |
| 8 | Prevenção de SQL Injection | `cursor.callproc()` parametrizado |
| 9 | FK RESTRICT impede exclusão com dependências | Integridade referencial |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 1 — Configuração do ambiente  (~1 minuto)
# Execute esta célula primeiro e aguarde o ✅ final
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, configparser
from pathlib import Path

def run(cmd, **kw):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)

# 1. Instalar MySQL Server
print('📦 Instalando MySQL Server (aguarde)...')
run('apt-get install -y mysql-server > /dev/null 2>&1')
run('service mysql start')

# 2. Configurar senha root
run("mysql -e \"ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'hotel123'; FLUSH PRIVILEGES;\"")
print('✅ MySQL configurado')

# 3. Clonar repositório
if not Path('hotel-mosquito-labdb').exists():
    print('📂 Clonando repositório...')
    run('git clone https://github.com/Matheus-PC-Silva/hotel-mosquito-labdb.git -q')
else:
    print('📂 Repositório já existe, atualizando...')
    run('git -C hotel-mosquito-labdb pull -q')
print('✅ Repositório pronto')

# 4. Instalar dependência Python
print('📦 Instalando mysql-connector-python...')
run(f'{sys.executable} -m pip install mysql-connector-python==8.4.0 -q')
print('✅ Pacote instalado')

# 5. Ajustar config.ini para usar porta 3306 (MySQL local, não Docker)
cfg_path = Path('hotel-mosquito-labdb/app/config.ini')
cfg = configparser.ConfigParser()
cfg.read(cfg_path)
cfg['database']['port'] = '3306'
cfg['database']['host'] = 'localhost'
cfg['database']['password'] = 'hotel123'
with open(cfg_path, 'w') as f:
    cfg.write(f)
print('✅ config.ini ajustado para MySQL local')

# 6. Adicionar projeto ao path
proj = str(Path('hotel-mosquito-labdb').resolve())
if proj not in sys.path:
    sys.path.insert(0, proj)

print('\n✅✅✅ Ambiente pronto! Execute a próxima célula.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar banco de dados
# ══════════════════════════════════════════════════════════════════
import subprocess
from pathlib import Path

sql_file = Path('hotel-mosquito-labdb/sql/hotel_mosquito_full.sql')
print('🗄️  Criando schema, views, procedures, triggers e dados de exemplo...')

result = subprocess.run(
    ['mysql', '-uroot', '-photel123', '--default-character-set=utf8mb4'],
    input=sql_file.read_bytes(),
    capture_output=True
)

if result.returncode == 0:
    print('✅ Banco carregado!\n')
else:
    print('❌ Erro:', result.stderr.decode(errors='replace'))
    raise SystemExit('Pare aqui e verifique o erro acima.')

# Verificar contagem dos dados
check = subprocess.run(
    ['mysql', '-uroot', '-photel123', 'hotel_mosquito', '-t', '-e',
     'SELECT '
     '(SELECT COUNT(*) FROM Categoria_Quarto) AS categorias,'
     '(SELECT COUNT(*) FROM Quarto)           AS quartos,'
     '(SELECT COUNT(*) FROM Cliente)          AS clientes,'
     '(SELECT COUNT(*) FROM Funcionario)      AS funcionarios,'
     '(SELECT COUNT(*) FROM Produto_Servico)  AS produtos,'
     '(SELECT COUNT(*) FROM Reserva)          AS reservas,'
     '(SELECT COUNT(*) FROM Hospedagem)       AS hospedagens,'
     '(SELECT COUNT(*) FROM Consumo)          AS consumos;'],
    capture_output=True, text=True
)
print('📊 Contagem esperada: 5 | 15 | 10 | 5 | 10 | 10 | 8 | 15')
print(check.stdout)

# Verificar procedures
procs = subprocess.run(
    ['mysql', '-uroot', '-photel123', 'hotel_mosquito', '-N', '-e',
     'SELECT COUNT(*) FROM information_schema.ROUTINES '
     'WHERE ROUTINE_SCHEMA="hotel_mosquito" AND ROUTINE_TYPE="PROCEDURE";'],
    capture_output=True, text=True
)
print(f'📋 Stored Procedures criadas: {procs.stdout.strip()} (esperado: ~26)')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 3 — Conectar e importar módulos
# ══════════════════════════════════════════════════════════════════
from app.db.connection import connect, call_proc
from app.db import auth, clientes, quartos, reservas, hospedagens, relatorios

conn = connect()
print('✅ Conexão com MySQL estabelecida!')
print(f'   Host: localhost:3306 | Banco: hotel_mosquito\n')

# Login dos 5 funcionários disponíveis
print('👥 Funcionários disponíveis:')
for login, perfil in [('gerente1','Gerente'),('gerente2','Gerente'),
                       ('recep1','Recepcionista'),('recep2','Recepcionista'),
                       ('recep3','Recepcionista')]:
    print(f'   {login:10s} | senha123 | {perfil}')

---
## 🧪 Testes

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 1 — Autenticação (sp_Login)
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('TESTE 1 — Autenticação')
print('=' * 60)

# 1a. Login correto — Gerente
gerente = auth.login(conn, 'gerente1', 'senha123')
print(f'\n✅ Login gerente1/senha123:')
print(f'   Nome:   {gerente["nome"]}')
print(f'   Perfil: {gerente["perfil"]}')
print(f'   ID:     {gerente["id_funcionario"]}')

# 1b. Login correto — Recepcionista
recep = auth.login(conn, 'recep1', 'senha123')
print(f'\n✅ Login recep1/senha123:')
print(f'   Nome:   {recep["nome"]}')
print(f'   Perfil: {recep["perfil"]}')

# 1c. Login incorreto
print('\n🔒 Testando senha errada (gerente1 / senha_errada):')
try:
    auth.login(conn, 'gerente1', 'senha_errada')
    print('   ❌ FALHA: deveria ter bloqueado!')
except RuntimeError as e:
    print(f'   ✅ Bloqueado corretamente: "{e}"')

# 1d. Usuário inexistente
print('\n🔒 Testando usuário inexistente (fantasma / senha123):')
try:
    auth.login(conn, 'fantasma', 'senha123')
    print('   ❌ FALHA: deveria ter bloqueado!')
except RuntimeError as e:
    print(f'   ✅ Bloqueado corretamente: "{e}"')

print('\n📝 NOTA: A procedure nunca revela se o login existe ou não —')
print('         ambos retornam a mesma mensagem genérica.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 2 — Fluxo completo de hospedagem
#   Reserva → Check-in → Lançar Consumo → Check-out
# ══════════════════════════════════════════════════════════════════
from datetime import date, timedelta

print('=' * 60)
print('TESTE 2 — Fluxo completo de hospedagem')
print('=' * 60)

# Datas: checkin amanhã, checkout em 3 dias
hoje = date.today()
checkin  = (hoje + timedelta(days=1)).strftime('%Y-%m-%d')
checkout = (hoje + timedelta(days=4)).strftime('%Y-%m-%d')

# Buscar um quarto disponível
disponíveis = quartos.disponiveis(conn)
if not disponíveis:
    print('⚠️  Nenhum quarto disponível — use a aba Quartos para liberar um')
else:
    q = disponíveis[0]
    print(f'\n🏠 Quarto selecionado: {q["numero"]} | {q["categoria_nome"]} | R$ {q["preco_praticado"]}/dia')

    # Buscar um cliente
    clientes_lista = clientes.listar(conn)
    cliente = clientes_lista[0]
    print(f'👤 Cliente: {cliente[1]} (CPF: {cliente[2]})')

    # 2a. Registrar reserva
    print(f'\n📅 Registrando reserva ({checkin} → {checkout})...')
    try:
        id_reserva = reservas.registrar(
            conn, cliente[0], q['id_quarto'],
            gerente['id_funcionario'], checkin, checkout
        )
        print(f'   ✅ Reserva #{id_reserva} criada com status=Confirmada')
    except RuntimeError as e:
        print(f'   ⚠️  {e} (pode já existir reserva sobreposta — use outro quarto)')
        id_reserva = None

    if id_reserva:
        # 2b. Realizar Check-in
        print('\n🔑 Realizando check-in...')
        try:
            id_hosp = hospedagens.realizar_checkin(
                conn, id_reserva, recep['id_funcionario'])
            print(f'   ✅ Check-in realizado! Hospedagem #{id_hosp}')
            print(f'   Reserva #{id_reserva} → status=Efetivada')
            print(f'   Quarto {q["numero"]} → status=Ocupado')
        except RuntimeError as e:
            print(f'   ⚠️  {e}')
            id_hosp = None

        if id_hosp:
            # 2c. Lançar consumo
            print('\n🧾 Lançando consumo (produto 1, qtd 2)...')
            try:
                hospedagens.lancar_consumo(conn, id_hosp, 1, 2)
                print(f'   ✅ Consumo lançado com snapshot de preço')
            except RuntimeError as e:
                print(f'   ⚠️  {e}')

            # 2d. Realizar Check-out
            print('\n🏁 Realizando check-out...')
            try:
                resumo = hospedagens.realizar_checkout(
                    conn, id_hosp, recep['id_funcionario'])
                print(f'   ✅ Check-out realizado!')
                print(f'   Diárias:       {resumo["quantidade_diarias"]}  ×  R$ {resumo["preco_diaria_aplicado"]}')
                print(f'   Total diárias: R$ {resumo["total_diarias"]}')
                print(f'   Total consumo: R$ {resumo["total_consumo"]}')
                print(f'   ══ VALOR TOTAL: R$ {resumo["valor_total"]} ══')
                print(f'   Quarto {q["numero"]} → status=Limpeza')
            except RuntimeError as e:
                print(f'   ⚠️  {e}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 3 — Sobreposição de reservas (sp_RegistrarReserva)
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('TESTE 3 — Detecção de sobreposição de reservas')
print('=' * 60)

# Buscar uma reserva Confirmada existente
reservas_conf = reservas.listar(conn, status='Confirmada')
if reservas_conf:
    r = reservas_conf[0]
    print(f'\n📋 Reserva existente: #{r[0]}')
    print(f'   Quarto: {r[4]} | Período: {r[7]} → {r[8]}')

    # Tentar reservar o mesmo quarto no mesmo período
    print('\n🔒 Tentando criar reserva sobreposta no mesmo quarto e período:')
    try:
        reservas.registrar(
            conn, cliente[0], r[3],  # mesmo quarto
            gerente['id_funcionario'], str(r[7]), str(r[8])
        )
        print('   ❌ FALHA: deveria ter bloqueado!')
    except RuntimeError as e:
        print(f'   ✅ Bloqueado pelo banco: "{e}"')
        print('   ROLLBACK automático via EXIT HANDLER')
else:
    print('\nℹ️  Sem reservas Confirmadas no momento (todas já efetivadas ou canceladas)')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 4 — Triggers: histórico automático de preço
# ══════════════════════════════════════════════════════════════════
import subprocess

def query(sql):
    r = subprocess.run(
        ['mysql', '-uroot', '-photel123', 'hotel_mosquito', '-t', '-e', sql],
        capture_output=True, text=True
    )
    return r.stdout

print('=' * 60)
print('TESTE 4 — Triggers (trg_preco_inicial + trg_atualiza_preco)')
print('=' * 60)

print('\n📊 Histórico atual do Quarto #1:')
print(query('SELECT id_historico, preco_praticado, data_inicio_vigencia, '
            'IFNULL(data_fim_vigencia, "(vigente)") AS fim '
            'FROM Historico_Preco WHERE id_quarto=1 ORDER BY id_historico;'))

print('\n💰 Alterando preço do Quarto #1 (via sp_AtualizarPrecoQuarto)...')
try:
    quartos.atualizar_preco(conn, 1, 399.99, gerente['id_funcionario'])
    print('   ✅ Preço atualizado para R$ 399.99')
except RuntimeError as e:
    print(f'   Erro: {e}')

print('\n📊 Histórico APÓS alteração:')
print(query('SELECT id_historico, preco_praticado, data_inicio_vigencia, '
            'IFNULL(data_fim_vigencia, "(vigente)") AS fim '
            'FROM Historico_Preco WHERE id_quarto=1 ORDER BY id_historico;'))
print('📝 O trigger fechou a vigência anterior e abriu nova — automaticamente.')

print('\n🔧 Testando trigger via UPDATE direto (simulando Workbench):')
query('UPDATE Quarto SET preco_praticado = 450.00 WHERE id_quarto = 1')
print('   UPDATE executado diretamente na tabela')
print('\n📊 Histórico após UPDATE direto:')
print(query('SELECT id_historico, preco_praticado, data_inicio_vigencia, '
            'IFNULL(data_fim_vigencia, "(vigente)") AS fim '
            'FROM Historico_Preco WHERE id_quarto=1 ORDER BY id_historico;'))
print('📝 Trigger disparou mesmo sem usar a procedure — invariante garantida pelo SGBD.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 5 — Transactions: ROLLBACK desfaz trigger
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('TESTE 5 — Transactions: ROLLBACK')
print('=' * 60)

print('\n📊 Estado ANTES da transação:')
print(query('SELECT preco_praticado FROM Quarto WHERE id_quarto=1;'))
n_hist_antes = query('SELECT COUNT(*) AS total FROM Historico_Preco WHERE id_quarto=1;')
print('Linhas em Historico_Preco (quarto 1):', n_hist_antes.strip())

print('\n🔄 Iniciando transação com UPDATE de preço...')
cursor = conn.cursor()
conn.autocommit = False
cursor.execute('UPDATE Quarto SET preco_praticado = 9999.00 WHERE id_quarto = 1')
print('   UPDATE executado (trigger criou nova linha em Historico_Preco)')

print('\n📊 Estado DENTRO da transação (antes do ROLLBACK):')
cursor.execute('SELECT preco_praticado FROM Quarto WHERE id_quarto=1')
print(f'   Preço atual (uncommitted): R$ {cursor.fetchone()[0]}')

print('\n↩️  Executando ROLLBACK...')
conn.rollback()
conn.autocommit = True
cursor.close()

print('\n📊 Estado APÓS o ROLLBACK:')
print(query('SELECT preco_praticado FROM Quarto WHERE id_quarto=1;'))
n_hist_depois = query('SELECT COUNT(*) AS total FROM Historico_Preco WHERE id_quarto=1;')
print('Linhas em Historico_Preco (quarto 1):', n_hist_depois.strip())
print('\n📝 ROLLBACK desfez o UPDATE E o INSERT do trigger — tudo na mesma transação.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 6 — Transactions: SAVEPOINT com rollback parcial
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('TESTE 6 — Transactions: SAVEPOINT')
print('=' * 60)

cursor = conn.cursor()
conn.autocommit = False

print('\n💰 Preços antes da transação:')
cursor.execute('SELECT id_quarto, numero, preco_praticado FROM Quarto WHERE id_quarto IN (2,3)')
for row in cursor.fetchall():
    print(f'   Quarto {row[1]} (id={row[0]}): R$ {row[2]}')

print('\n🔄 START TRANSACTION')
cursor.execute('UPDATE Quarto SET preco_praticado = 200.00 WHERE id_quarto = 2')
print('   UPDATE quarto #2 → R$ 200,00')

cursor.execute('SAVEPOINT sp_quarto2')
print('   SAVEPOINT sp_quarto2')

cursor.execute('UPDATE Quarto SET preco_praticado = 300.00 WHERE id_quarto = 3')
print('   UPDATE quarto #3 → R$ 300,00')

print('\n↩️  ROLLBACK TO SAVEPOINT sp_quarto2 (só desfaz o quarto #3)')
cursor.execute('ROLLBACK TO SAVEPOINT sp_quarto2')

print('\n✅ COMMIT (persiste quarto #2, descarta quarto #3)')
conn.commit()
conn.autocommit = True

print('\n💰 Preços APÓS a transação:')
cursor.execute('SELECT id_quarto, numero, preco_praticado FROM Quarto WHERE id_quarto IN (2,3)')
for row in cursor.fetchall():
    print(f'   Quarto {row[1]} (id={row[0]}): R$ {row[2]}')
cursor.close()
print('\n📝 Quarto #2 atualizado; quarto #3 voltou ao valor original.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 7 — Controle de acesso RBAC (sp_AssertGerente)
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('TESTE 7 — Controle de acesso (RBAC)')
print('=' * 60)

id_ger  = gerente['id_funcionario']
id_rec  = recep['id_funcionario']

# 7a. Gerente pode alterar preço
print(f'\n👔 Gerente (id={id_ger}) altera preço do quarto #1:')
try:
    quartos.atualizar_preco(conn, 1, 350.00, id_ger)
    print('   ✅ Permitido — gerente autorizado')
except RuntimeError as e:
    print(f'   ❌ {e}')

# 7b. Recepcionista NÃO pode alterar preço
print(f'\n👷 Recepcionista (id={id_rec}) tenta alterar preço do quarto #1:')
try:
    quartos.atualizar_preco(conn, 1, 9999.00, id_rec)
    print('   ❌ FALHA: deveria ter bloqueado!')
except RuntimeError as e:
    print(f'   ✅ Bloqueado pelo banco: "{e}"')

# 7c. Recepcionista NÃO pode alterar categoria
print(f'\n👷 Recepcionista (id={id_rec}) tenta alterar categoria do quarto #1:')
try:
    quartos.atualizar_categoria(conn, 1, 2, id_rec)
    print('   ❌ FALHA: deveria ter bloqueado!')
except RuntimeError as e:
    print(f'   ✅ Bloqueado pelo banco: "{e}"')

# 7d. Recepcionista NÃO pode acessar relatórios
print(f'\n👷 Recepcionista (id={id_rec}) tenta acessar relatório de faturamento:')
try:
    relatorios.faturamento_mensal(conn, id_rec)
    print('   ❌ FALHA: deveria ter bloqueado!')
except RuntimeError as e:
    print(f'   ✅ Bloqueado pelo banco: "{e}"')

# 7e. Gerente PODE acessar relatórios
print(f'\n👔 Gerente (id={id_ger}) acessa relatório de faturamento:')
try:
    fat = relatorios.faturamento_mensal(conn, id_ger)
    print(f'   ✅ Acesso permitido — {len(fat)} mês(es) de dados')
except RuntimeError as e:
    print(f'   Erro: {e}')

print('\n📝 O bloqueio é no BANCO — funciona mesmo via acesso direto ao MySQL.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 8 — Prevenção de SQL Injection
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('TESTE 8 — Prevenção de SQL Injection')
print('=' * 60)

# Tentativa clássica de SQL Injection no campo de login
payloads = [
    ("' OR '1'='1",      'senha_qualquer'),   # bypass de autenticação
    ("admin'--",          'senha_qualquer'),   # comentário SQL
    ("'; DROP TABLE Funcionario;--", 'x'),     # tentativa destrutiva
]

print('\n🔒 Testando payloads de SQL Injection no sp_Login:\n')
for login_payload, senha in payloads:
    try:
        auth.login(conn, login_payload, senha)
        print(f'   ❌ VULNERÁVEL! Payload: {login_payload!r}')
    except RuntimeError as e:
        print(f'   ✅ Bloqueado: {login_payload!r}')
        print(f'      Resposta: "{e}"')

print('\n📊 Verificando que a tabela Funcionario ainda existe:')
r = subprocess.run(
    ['mysql', '-uroot', '-photel123', 'hotel_mosquito', '-N', '-e',
     'SELECT COUNT(*) FROM Funcionario;'],
    capture_output=True, text=True
)
print(f'   Funcionários no banco: {r.stdout.strip()} (esperado: 5) ✅')

print()
print('📝 Por que funciona:')
print('   cursor.callproc("sp_Login", (login, senha))')
print('   Os valores são enviados como DADOS, não como código SQL.')
print('   O MySQL os recebe como strings literais — injetável seria impossível.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TESTE 9 — Integridade referencial (FK RESTRICT)
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('TESTE 9 — Integridade Referencial (FK RESTRICT)')
print('=' * 60)

# Tentar excluir um cliente que possui reservas
print('\n🗑️  Tentando excluir cliente #1 (possui reservas vinculadas):')
try:
    clientes.excluir(conn, 1)
    print('   ❌ FALHA: deveria ter bloqueado!')
except RuntimeError as e:
    print(f'   ✅ Bloqueado pela FK: "{e}"')

# Tentar excluir um quarto que possui reservas
print('\n🗑️  Tentando excluir quarto #1 (possui reservas vinculadas):')
try:
    quartos.excluir(conn, 1)
    print('   ❌ FALHA: deveria ter bloqueado!')
except RuntimeError as e:
    print(f'   ✅ Bloqueado pela FK: "{e}"')

print('\n📝 ON DELETE RESTRICT impede exclusão de registros com dependentes,')
print('   preservando o histórico de reservas e hospedagens.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESUMO FINAL
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('✅ TODOS OS TESTES CONCLUÍDOS')
print('=' * 60)
print()
print('Elemento BD         | Resultado')
print('-' * 50)
print('sp_Login            | ✅ Autenticação + bloqueio')
print('SPs transacionais   | ✅ Reserva → Check-in → Consumo → Check-out')
print('Sobreposição        | ✅ SIGNAL + ROLLBACK automático')
print('Trigger AFTER INSERT| ✅ Histórico criado no nascimento do quarto')
print('Trigger AFTER UPDATE| ✅ Vigência fechada + nova aberta')
print('ROLLBACK            | ✅ Desfaz trigger + UPDATE')
print('SAVEPOINT           | ✅ Rollback parcial confirmado')
print('sp_AssertGerente    | ✅ Bloqueio em 4 operações diferentes')
print('SQL Injection       | ✅ 3 payloads bloqueados')
print('FK RESTRICT         | ✅ Integridade referencial garantida')
print()
print('🏨 Hotel do Mosquito — LABDB')

conn.close()